In [0]:
# ============================================================
# SILVER TRANSFORMATION 1
# DUPLICATE RECORD REMOVAL
# ============================================================

from pyspark.sql import functions as F

# Create Silver schema
spark.sql("CREATE SCHEMA IF NOT EXISTS credit_risk_silver")


# ------------------------------------------------------------
# 1. APPLICATION(S)
# ------------------------------------------------------------

df = spark.table("credit_risk_bronze.`application(s)`")

df = df.dropDuplicates()

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("credit_risk_silver.`application(s)`")


# ------------------------------------------------------------
# 2. BUREAU
# ------------------------------------------------------------

df = spark.table("credit_risk_bronze.bureau")

df = df.dropDuplicates()

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("credit_risk_silver.bureau")


# ------------------------------------------------------------
# 3. BUREAU BALANCE
# ------------------------------------------------------------

df = spark.table("credit_risk_bronze.bureau_balance")

df = df.dropDuplicates()

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("credit_risk_silver.bureau_balance")


# ------------------------------------------------------------
# 4. CREDIT CARD BALANCE
# ------------------------------------------------------------

df = spark.table("credit_risk_bronze.credit_card_balance")

df = df.dropDuplicates()

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("credit_risk_silver.credit_card_balance")


# ------------------------------------------------------------
# 5. INSTALLMENTS PAYMENTS
# ------------------------------------------------------------

df = spark.table("credit_risk_bronze.installments_payments")

df = df.dropDuplicates()

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("credit_risk_silver.installments_payments")


# ------------------------------------------------------------
# 6. POS CASH BALANCE
# ------------------------------------------------------------

df = spark.table("credit_risk_bronze.POS_CASH_balance")

df = df.dropDuplicates()

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("credit_risk_silver.POS_CASH_balance")


# ------------------------------------------------------------
# 7. PREVIOUS APPLICATION
# ------------------------------------------------------------

df = spark.table("credit_risk_bronze.previous_application")

df = df.dropDuplicates()

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("credit_risk_silver.previous_application")


print("Transformation 1 - Duplicate Removal completed successfully.")

Transformation 1 - Duplicate Removal completed successfully.


In [0]:
# ============================================================
# VALIDATION - BRONZE VS SILVER ROW COUNTS
# ============================================================

tables = [
    "application(s)",
    "bureau",
    "bureau_balance",
    "credit_card_balance",
    "installments_payments",
    "POS_CASH_balance",
    "previous_application"
]

for table in tables:

    bronze_count = spark.table(
        f"credit_risk_bronze.`{table}`"
    ).count()

    silver_count = spark.table(
        f"credit_risk_silver.`{table}`"
    ).count()

    difference = bronze_count - silver_count

    print(
        f"{table}: "
        f"Bronze = {bronze_count:,} | "
        f"Silver = {silver_count:,} | "
        f"Duplicates Removed = {difference:,}"
    )

application(s): Bronze = 307,521 | Silver = 307,511 | Duplicates Removed = 10
bureau: Bronze = 1,716,438 | Silver = 1,716,428 | Duplicates Removed = 10
bureau_balance: Bronze = 27,299,925 | Silver = 27,299,925 | Duplicates Removed = 0
credit_card_balance: Bronze = 3,840,312 | Silver = 3,840,312 | Duplicates Removed = 0
installments_payments: Bronze = 13,605,401 | Silver = 13,605,401 | Duplicates Removed = 0
POS_CASH_balance: Bronze = 10,001,358 | Silver = 10,001,358 | Duplicates Removed = 0
previous_application: Bronze = 1,670,224 | Silver = 1,670,214 | Duplicates Removed = 10


In [0]:
# ============================================================
# MISSING VALUE ANALYSIS
# ============================================================

from pyspark.sql import functions as F

# ------------------------------------------------------------
# All Silver tables
# ------------------------------------------------------------

tables = [
    "application(s)",
    "bureau",
    "bureau_balance",
    "credit_card_balance",
    "installments_payments",
    "POS_CASH_balance",
    "previous_application"
]


# ------------------------------------------------------------
# Check NULL values in every column of every table
# ------------------------------------------------------------

for table in tables:

    print(f"\n{'=' * 70}")
    print(f"TABLE: {table}")
    print(f"{'=' * 70}")

    # Read Silver table
    df = spark.table(f"credit_risk_silver.`{table}`")

    # Total number of rows
    total_rows = df.count()

    missing_data = []

    # --------------------------------------------------------
    # Check every column
    # --------------------------------------------------------

    for column in df.columns:

        null_count = df.filter(
            F.col(column).isNull()
        ).count()

        # Only include columns having NULL values
        if null_count > 0:

            null_percentage = (
                null_count / total_rows
            ) * 100

            data_type = str(
                df.schema[column].dataType
            )

            missing_data.append(
                (
                    table,
                    column,
                    data_type,
                    total_rows,
                    null_count,
                    round(null_percentage, 2)
                )
            )

    # --------------------------------------------------------
    # Display missing-value results
    # --------------------------------------------------------

    if missing_data:

        result_df = spark.createDataFrame(
            missing_data,
            [
                "table_name",
                "column_name",
                "data_type",
                "total_rows",
                "null_count",
                "null_percentage"
            ]
        )

        display(
            result_df.orderBy(
                F.col("null_percentage").desc()
            )
        )

    else:

        print("No NULL values found in this table.")


print("\n")
print("=" * 70)
print("MISSING VALUE ANALYSIS COMPLETED")
print("=" * 70)


TABLE: application(s)


table_name,column_name,data_type,total_rows,null_count,null_percentage
application(s),_rescued_data,StringType(),307511,307511,100.0
application(s),COMMONAREA_AVG,DoubleType(),307511,214865,69.87
application(s),COMMONAREA_MODE,DoubleType(),307511,214865,69.87
application(s),COMMONAREA_MEDI,DoubleType(),307511,214865,69.87
application(s),NONLIVINGAPARTMENTS_AVG,DoubleType(),307511,213514,69.43
application(s),NONLIVINGAPARTMENTS_MODE,DoubleType(),307511,213514,69.43
application(s),NONLIVINGAPARTMENTS_MEDI,DoubleType(),307511,213514,69.43
application(s),FONDKAPREMONT_MODE,StringType(),307511,210295,68.39
application(s),LIVINGAPARTMENTS_AVG,DoubleType(),307511,210199,68.35
application(s),LIVINGAPARTMENTS_MODE,DoubleType(),307511,210199,68.35



TABLE: bureau


table_name,column_name,data_type,total_rows,null_count,null_percentage
bureau,_rescued_data,StringType(),1716428,1716428,100.0
bureau,AMT_ANNUITY,DoubleType(),1716428,1226791,71.47
bureau,AMT_CREDIT_MAX_OVERDUE,DoubleType(),1716428,1124488,65.51
bureau,DAYS_ENDDATE_FACT,DoubleType(),1716428,633653,36.92
bureau,AMT_CREDIT_SUM_LIMIT,DoubleType(),1716428,591780,34.48
bureau,AMT_CREDIT_SUM_DEBT,DoubleType(),1716428,257669,15.01
bureau,DAYS_CREDIT_ENDDATE,DoubleType(),1716428,105553,6.15
bureau,AMT_CREDIT_SUM,DoubleType(),1716428,13,0.0



TABLE: bureau_balance


...


TABLE: POS_CASH_balance


table_name,column_name,data_type,total_rows,null_count,null_percentage
POS_CASH_balance,_rescued_data,StringType(),10001358,10001358,100.0
POS_CASH_balance,CNT_INSTALMENT,DoubleType(),10001358,26071,0.26
POS_CASH_balance,CNT_INSTALMENT_FUTURE,DoubleType(),10001358,26087,0.26



TABLE: previous_application


table_name,column_name,data_type,total_rows,null_count,null_percentage
previous_application,_rescued_data,StringType(),1670214,1670214,100.0
previous_application,RATE_INTEREST_PRIMARY,DoubleType(),1670214,1664263,99.64
previous_application,RATE_INTEREST_PRIVILEGED,DoubleType(),1670214,1664263,99.64
previous_application,AMT_DOWN_PAYMENT,DoubleType(),1670214,895844,53.64
previous_application,RATE_DOWN_PAYMENT,DoubleType(),1670214,895844,53.64
previous_application,NAME_TYPE_SUITE,StringType(),1670214,820405,49.12
previous_application,DAYS_FIRST_DRAWING,DoubleType(),1670214,673065,40.3
previous_application,DAYS_FIRST_DUE,DoubleType(),1670214,673065,40.3
previous_application,DAYS_LAST_DUE_1ST_VERSION,DoubleType(),1670214,673065,40.3
previous_application,DAYS_LAST_DUE,DoubleType(),1670214,673065,40.3




MISSING VALUE ANALYSIS COMPLETED


In [0]:
# ============================================================
# SILVER TRANSFORMATION 2
# HANDLE MISSING VALUES
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import NumericType, StringType


# ------------------------------------------------------------
# Tables for Transformation 2
# ------------------------------------------------------------

tables = [
    "application(s)",
    "bureau",
    "bureau_balance",
    "credit_card_balance",
    "installments_payments",
    "POS_CASH_balance",
    "previous_application"
]


# ------------------------------------------------------------
# Columns with very high missing values
# These will be handled in Transformation 3
# ------------------------------------------------------------

HIGH_MISSING_THRESHOLD = 50.0


# ------------------------------------------------------------
# Process every Silver table
# ------------------------------------------------------------

for table in tables:

    print(f"\nProcessing: {table}")

    df = spark.table(f"credit_risk_silver.`{table}`")

    total_rows = df.count()

    fill_values = {}

    # --------------------------------------------------------
    # Find suitable values for NULLs
    # --------------------------------------------------------

    for field in df.schema.fields:

        column = field.name

        # Count NULL values
        null_count = df.filter(
            F.col(column).isNull()
        ).count()

        if null_count == 0:
            continue

        missing_percentage = (
            null_count / total_rows
        ) * 100

        # ----------------------------------------------------
        # Highly missing columns are NOT handled here
        # They will be handled in Transformation 3
        # ----------------------------------------------------

        if missing_percentage >= HIGH_MISSING_THRESHOLD:

            print(
                f"Skipped for T3: {column} "
                f"({missing_percentage:.2f}% missing)"
            )

            continue

        # ----------------------------------------------------
        # NUMERIC COLUMN
        # Fill NULL with median
        # ----------------------------------------------------

        if isinstance(field.dataType, NumericType):

            median_value = df.approxQuantile(
                column,
                [0.5],
                0.01
            )[0]

            fill_values[column] = median_value

            print(
                f"Numeric -> {column}: "
                f"NULLs filled with median {median_value}"
            )

        # ----------------------------------------------------
        # STRING / CATEGORICAL COLUMN
        # Fill NULL with mode
        # ----------------------------------------------------

        elif isinstance(field.dataType, StringType):

            mode_row = (
                df
                .filter(F.col(column).isNotNull())
                .groupBy(column)
                .count()
                .orderBy(F.desc("count"))
                .first()
            )

            if mode_row is not None:

                fill_values[column] = mode_row[0]

                print(
                    f"Categorical -> {column}: "
                    f"NULLs filled with mode '{mode_row[0]}'"
                )

    # --------------------------------------------------------
    # Apply the missing-value treatment
    # --------------------------------------------------------

    if fill_values:

        df_cleaned = df.fillna(fill_values)

    else:

        df_cleaned = df

    # --------------------------------------------------------
    # Save transformed table back to Silver
    # --------------------------------------------------------

    df_cleaned.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(
            f"credit_risk_silver.`{table}`"
        )

    print(
        f"Completed: {table} | "
        f"Columns handled: {len(fill_values)}"
    )


print("\n" + "=" * 70)
print("TRANSFORMATION 2 - MISSING VALUE HANDLING COMPLETED")
print("=" * 70)


Processing: application(s)
Numeric -> AMT_ANNUITY: NULLs filled with median 25150.5
Numeric -> AMT_GOODS_PRICE: NULLs filled with median 450000.0
Categorical -> NAME_TYPE_SUITE: NULLs filled with mode 'Unaccompanied'
Skipped for T3: OWN_CAR_AGE (65.99% missing)
Categorical -> OCCUPATION_TYPE: NULLs filled with mode 'Laborers'
Numeric -> CNT_FAM_MEMBERS: NULLs filled with median 2.0
Skipped for T3: EXT_SOURCE_1 (56.38% missing)
Numeric -> EXT_SOURCE_2: NULLs filled with median 0.5640642411296688
Numeric -> EXT_SOURCE_3: NULLs filled with median 0.5352762504724826
Skipped for T3: APARTMENTS_AVG (50.75% missing)
Skipped for T3: BASEMENTAREA_AVG (58.52% missing)
Numeric -> YEARS_BEGINEXPLUATATION_AVG: NULLs filled with median 0.9821
Skipped for T3: YEARS_BUILD_AVG (66.50% missing)
Skipped for T3: COMMONAREA_AVG (69.87% missing)
Skipped for T3: ELEVATORS_AVG (53.30% missing)
Skipped for T3: ENTRANCES_AVG (50.35% missing)
Numeric -> FLOORSMAX_AVG: NULLs filled with median 0.1667
Skipped for

In [0]:
# ============================================================
# VALIDATION - TRANSFORMATION 2
# CHECK REMAINING NULL VALUES
# ============================================================

from pyspark.sql import functions as F

tables = [
    "application(s)",
    "bureau",
    "bureau_balance",
    "credit_card_balance",
    "installments_payments",
    "POS_CASH_balance",
    "previous_application"
]

HIGH_MISSING_THRESHOLD = 50.0

for table in tables:

    df = spark.table(f"credit_risk_silver.`{table}`")

    total_rows = df.count()

    remaining_nulls = []

    for column in df.columns:

        null_count = df.filter(
            F.col(column).isNull()
        ).count()

        if null_count > 0:

            null_percentage = (
                null_count / total_rows
            ) * 100

            remaining_nulls.append(
                (
                    table,
                    column,
                    null_count,
                    round(null_percentage, 2),
                    "T3 - Highly Missing"
                    if null_percentage >= HIGH_MISSING_THRESHOLD
                    else "Needs Review"
                )
            )

    if remaining_nulls:

        result_df = spark.createDataFrame(
            remaining_nulls,
            [
                "table_name",
                "column_name",
                "remaining_null_count",
                "remaining_null_percentage",
                "status"
            ]
        )

        print(f"\n========== {table} ==========")

        display(
            result_df.orderBy(
                F.desc("remaining_null_percentage")
            )
        )

    else:

        print(f"\n{table}: No NULL values remaining.")


print("\n" + "=" * 70)
print("TRANSFORMATION 2 VALIDATION COMPLETED")
print("=" * 70)


========== application(s) ==========


table_name,column_name,remaining_null_count,remaining_null_percentage,status
application(s),_rescued_data,307511,100.0,T3 - Highly Missing
application(s),COMMONAREA_AVG,214865,69.87,T3 - Highly Missing
application(s),COMMONAREA_MODE,214865,69.87,T3 - Highly Missing
application(s),COMMONAREA_MEDI,214865,69.87,T3 - Highly Missing
application(s),NONLIVINGAPARTMENTS_AVG,213514,69.43,T3 - Highly Missing
application(s),NONLIVINGAPARTMENTS_MODE,213514,69.43,T3 - Highly Missing
application(s),NONLIVINGAPARTMENTS_MEDI,213514,69.43,T3 - Highly Missing
application(s),FONDKAPREMONT_MODE,210295,68.39,T3 - Highly Missing
application(s),LIVINGAPARTMENTS_AVG,210199,68.35,T3 - Highly Missing
application(s),LIVINGAPARTMENTS_MODE,210199,68.35,T3 - Highly Missing



========== bureau ==========


table_name,column_name,remaining_null_count,remaining_null_percentage,status
bureau,_rescued_data,1716428,100.0,T3 - Highly Missing
bureau,AMT_ANNUITY,1226791,71.47,T3 - Highly Missing
bureau,AMT_CREDIT_MAX_OVERDUE,1124488,65.51,T3 - Highly Missing



========== bureau_balance ==========


...


========== POS_CASH_balance ==========


table_name,column_name,remaining_null_count,remaining_null_percentage,status
POS_CASH_balance,_rescued_data,10001358,100.0,T3 - Highly Missing



========== previous_application ==========


table_name,column_name,remaining_null_count,remaining_null_percentage,status
previous_application,_rescued_data,1670214,100.0,T3 - Highly Missing
previous_application,RATE_INTEREST_PRIMARY,1664263,99.64,T3 - Highly Missing
previous_application,RATE_INTEREST_PRIVILEGED,1664263,99.64,T3 - Highly Missing
previous_application,AMT_DOWN_PAYMENT,895844,53.64,T3 - Highly Missing
previous_application,RATE_DOWN_PAYMENT,895844,53.64,T3 - Highly Missing



TRANSFORMATION 2 VALIDATION COMPLETED


In [0]:
# ============================================================
# TRANSFORMATION 3 - HANDLE HIGHLY MISSING COLUMNS
# ============================================================

from pyspark.sql import functions as F

tables = [
    "application(s)",
    "bureau",
    "bureau_balance",
    "credit_card_balance",
    "installments_payments",
    "POS_CASH_balance",
    "previous_application"
]

HIGH_MISSING_THRESHOLD = 50.0

# Important identifier columns - never remove these
key_columns = {
    "SK_ID_CURR",
    "SK_ID_BUREAU",
    "SK_ID_PREV",
    "SK_ID_CREDIT_CARD",
    "SK_ID_INSTALMENT",
    "SK_ID_POS_CASH"
}

for table in tables:

    df = spark.table(f"credit_risk_silver.`{table}`")

    total_rows = df.count()

    columns_to_drop = []

    for column in df.columns:

        null_count = df.filter(
            F.col(column).isNull()
        ).count()

        missing_percentage = (
            null_count / total_rows
        ) * 100

        # Drop highly missing non-key columns
        if (
            missing_percentage >= HIGH_MISSING_THRESHOLD
            and column not in key_columns
        ):
            columns_to_drop.append(column)

    if columns_to_drop:

        print(f"\n{table}")
        print("Columns removed:")

        for column in columns_to_drop:
            print(f"  - {column}")

        df = df.drop(*columns_to_drop)

        # Save transformed table
        df.write \
          .format("delta") \
          .mode("overwrite") \
          .option("overwriteSchema", "true") \
          .saveAsTable(f"credit_risk_silver.`{table}`")

    else:

        print(f"\n{table}: No highly missing columns to remove.")


print("\n" + "=" * 70)
print("TRANSFORMATION 3 COMPLETED")
print("=" * 70)


application(s)
Columns removed:
  - OWN_CAR_AGE
  - EXT_SOURCE_1
  - APARTMENTS_AVG
  - BASEMENTAREA_AVG
  - YEARS_BUILD_AVG
  - COMMONAREA_AVG
  - ELEVATORS_AVG
  - ENTRANCES_AVG
  - FLOORSMIN_AVG
  - LANDAREA_AVG
  - LIVINGAPARTMENTS_AVG
  - LIVINGAREA_AVG
  - NONLIVINGAPARTMENTS_AVG
  - NONLIVINGAREA_AVG
  - APARTMENTS_MODE
  - BASEMENTAREA_MODE
  - YEARS_BUILD_MODE
  - COMMONAREA_MODE
  - ELEVATORS_MODE
  - ENTRANCES_MODE
  - FLOORSMIN_MODE
  - LANDAREA_MODE
  - LIVINGAPARTMENTS_MODE
  - LIVINGAREA_MODE
  - NONLIVINGAPARTMENTS_MODE
  - NONLIVINGAREA_MODE
  - APARTMENTS_MEDI
  - BASEMENTAREA_MEDI
  - YEARS_BUILD_MEDI
  - COMMONAREA_MEDI
  - ELEVATORS_MEDI
  - ENTRANCES_MEDI
  - FLOORSMIN_MEDI
  - LANDAREA_MEDI
  - LIVINGAPARTMENTS_MEDI
  - LIVINGAREA_MEDI
  - NONLIVINGAPARTMENTS_MEDI
  - NONLIVINGAREA_MEDI
  - FONDKAPREMONT_MODE
  - HOUSETYPE_MODE
  - WALLSMATERIAL_MODE
  - _rescued_data

bureau
Columns removed:
  - AMT_CREDIT_MAX_OVERDUE
  - AMT_ANNUITY
  - _rescued_data

bureau_b

In [0]:
# ============================================================
# VALIDATION - TRANSFORMATION 3
# CHECK HIGHLY MISSING COLUMNS WERE REMOVED
# ============================================================

from pyspark.sql import functions as F

tables = [
    "application(s)",
    "bureau",
    "bureau_balance",
    "credit_card_balance",
    "installments_payments",
    "POS_CASH_balance",
    "previous_application"
]

HIGH_MISSING_THRESHOLD = 50.0

for table in tables:

    df = spark.table(f"credit_risk_silver.`{table}`")

    total_rows = df.count()

    high_missing_columns = []

    for column in df.columns:

        null_count = df.filter(
            F.col(column).isNull()
        ).count()

        missing_percentage = (
            null_count / total_rows
        ) * 100

        if missing_percentage >= HIGH_MISSING_THRESHOLD:
            high_missing_columns.append(
                (
                    column,
                    null_count,
                    round(missing_percentage, 2)
                )
            )

    print(f"\n========== {table} ==========")

    print(f"Rows: {total_rows}")
    print(f"Columns: {len(df.columns)}")

    if high_missing_columns:

        print("Remaining highly missing columns:")

        for column, null_count, percentage in high_missing_columns:
            print(
                f"  - {column}: "
                f"{null_count} NULLs ({percentage}%)"
            )

    else:
        print("PASS - No highly missing columns remain.")


print("\n" + "=" * 70)
print("TRANSFORMATION 3 VALIDATION COMPLETED")
print("=" * 70)


========== application(s) ==========
Rows: 307511
Columns: 81
PASS - No highly missing columns remain.

========== bureau ==========
Rows: 1716428
Columns: 15
PASS - No highly missing columns remain.

========== bureau_balance ==========
Rows: 27299925
Columns: 3
PASS - No highly missing columns remain.

========== credit_card_balance ==========
Rows: 3840312
Columns: 23
PASS - No highly missing columns remain.

========== installments_payments ==========
Rows: 13605401
Columns: 8
PASS - No highly missing columns remain.

========== POS_CASH_balance ==========
Rows: 10001358
Columns: 8
PASS - No highly missing columns remain.

========== previous_application ==========
Rows: 1670214
Columns: 33
PASS - No highly missing columns remain.

TRANSFORMATION 3 VALIDATION COMPLETED


In [0]:
# ============================================================
# HANDLE REMAINING MISSING VALUES
# After T2 + T3
# ============================================================

from pyspark.sql import functions as F

tables = [
    "application(s)",
    "bureau",
    "bureau_balance",
    "credit_card_balance",
    "installments_payments",
    "POS_CASH_balance",
    "previous_application"
]

# Identifier columns - do NOT artificially fill these
key_columns = {
    "SK_ID_CURR",
    "SK_ID_BUREAU",
    "SK_ID_PREV",
    "SK_ID_CREDIT_CARD",
    "SK_ID_INSTALMENT",
    "SK_ID_POS_CASH"
}

for table in tables:

    print(f"\n========== {table} ==========")

    df = spark.table(f"credit_risk_silver.`{table}`")

    handled_columns = []

    for column in df.columns:

        # Skip identifier columns
        if column in key_columns:
            continue

        null_count = df.filter(
            F.col(column).isNull()
        ).count()

        if null_count == 0:
            continue

        # ----------------------------------------------------
        # NUMERIC COLUMN → MEDIAN
        # ----------------------------------------------------
        if dict(df.dtypes)[column] in [
            "int",
            "bigint",
            "double",
            "float",
            "long",
            "short",
            "decimal"
        ]:

            median_value = df.approxQuantile(
                column,
                [0.5],
                0.01
            )[0]

            if median_value is not None:

                df = df.fillna(
                    {column: median_value}
                )

                handled_columns.append(
                    (column, "MEDIAN", median_value)
                )

        # ----------------------------------------------------
        # STRING / CATEGORICAL COLUMN → MODE
        # ----------------------------------------------------
        elif dict(df.dtypes)[column] == "string":

            mode_row = (
                df.filter(F.col(column).isNotNull())
                  .groupBy(column)
                  .count()
                  .orderBy(F.desc("count"))
                  .first()
            )

            if mode_row is not None:

                mode_value = mode_row[0]

                df = df.fillna(
                    {column: mode_value}
                )

                handled_columns.append(
                    (column, "MODE", mode_value)
                )

    # Save updated Silver table
    df.write \
      .format("delta") \
      .mode("overwrite") \
      .option("overwriteSchema", "true") \
      .saveAsTable(f"credit_risk_silver.`{table}`")

    # Print summary
    if handled_columns:

        print("Missing values handled:")

        for column, method, value in handled_columns:
            print(
                f"  - {column} → {method} → {value}"
            )

    else:
        print("No remaining missing values to handle.")


print("\n" + "=" * 70)
print("REMAINING MISSING VALUES HANDLED")
print("=" * 70)


========== application(s) ==========
No remaining missing values to handle.

========== bureau ==========
No remaining missing values to handle.

========== bureau_balance ==========
No remaining missing values to handle.

========== credit_card_balance ==========
No remaining missing values to handle.

========== installments_payments ==========
No remaining missing values to handle.

========== POS_CASH_balance ==========
No remaining missing values to handle.

========== previous_application ==========
No remaining missing values to handle.

REMAINING MISSING VALUES HANDLED


In [0]:
# ============================================================
# VALIDATION - REMAINING MISSING VALUES
# ============================================================

from pyspark.sql import functions as F

tables = [
    "application(s)",
    "bureau",
    "bureau_balance",
    "credit_card_balance",
    "installments_payments",
    "POS_CASH_balance",
    "previous_application"
]

for table in tables:

    df = spark.table(f"credit_risk_silver.`{table}`")

    # Count all NULL values across all columns
    null_counts = [
        F.sum(
            F.when(F.col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in df.columns
    ]

    result = df.select(null_counts).collect()[0].asDict()

    total_nulls = sum(
        value for value in result.values()
        if value is not None
    )

    print(f"\n========== {table} ==========")
    print(f"Total NULL values: {total_nulls}")

    if total_nulls == 0:
        print("PASS - No NULL values remain.")
    else:
        print("FAIL - NULL values still remain.")

print("\n" + "=" * 70)
print("MISSING VALUE VALIDATION COMPLETED")
print("=" * 70)


========== application(s) ==========
Total NULL values: 0
PASS - No NULL values remain.

========== bureau ==========
Total NULL values: 0
PASS - No NULL values remain.

========== bureau_balance ==========
Total NULL values: 0
PASS - No NULL values remain.

========== credit_card_balance ==========
Total NULL values: 0
PASS - No NULL values remain.

========== installments_payments ==========
Total NULL values: 0
PASS - No NULL values remain.

========== POS_CASH_balance ==========
Total NULL values: 0
PASS - No NULL values remain.

========== previous_application ==========
Total NULL values: 0
PASS - No NULL values remain.

MISSING VALUE VALIDATION COMPLETED


In [0]:
# ============================================================
# T4 - CHECK CURRENT DATA TYPES
# ============================================================

tables = [
    "application(s)",
    "bureau",
    "bureau_balance",
    "credit_card_balance",
    "installments_payments",
    "POS_CASH_balance",
    "previous_application"
]

for table in tables:

    print(f"\n{'=' * 70}")
    print(f"TABLE: {table}")
    print(f"{'=' * 70}")

    df = spark.table(f"credit_risk_silver.`{table}`")

    for column, dtype in df.dtypes:
        print(f"{column:45} -> {dtype}")


TABLE: application(s)
SK_ID_CURR                                    -> int
TARGET                                        -> int
NAME_CONTRACT_TYPE                            -> string
CODE_GENDER                                   -> string
FLAG_OWN_CAR                                  -> string
FLAG_OWN_REALTY                               -> string
CNT_CHILDREN                                  -> int
AMT_INCOME_TOTAL                              -> double
AMT_CREDIT                                    -> double
AMT_ANNUITY                                   -> double
AMT_GOODS_PRICE                               -> double
NAME_TYPE_SUITE                               -> string
NAME_INCOME_TYPE                              -> string
NAME_EDUCATION_TYPE                           -> string
NAME_FAMILY_STATUS                            -> string
NAME_HOUSING_TYPE                             -> string
REGION_POPULATION_RELATIVE                    -> double
DAYS_BIRTH                        

In [0]:
# ============================================================
# TRANSFORMATION 4 - CORRECT DATA TYPES
# ============================================================

from pyspark.sql import functions as F

# Columns that represent counts/flags and should be integers
integer_columns = {

    "application(s)": [
        "CNT_FAM_MEMBERS"
    ],

    "bureau": [],

    "bureau_balance": [],

    "credit_card_balance": [
        "CNT_DRAWINGS_ATM_CURRENT",
        "CNT_DRAWINGS_OTHER_CURRENT",
        "CNT_DRAWINGS_POS_CURRENT",
        "CNT_INSTALMENT_MATURE_CUM"
    ],

    "installments_payments": [],

    "POS_CASH_balance": [
        "CNT_INSTALMENT",
        "CNT_INSTALMENT_FUTURE"
    ],

    "previous_application": [
        "CNT_PAYMENT",
        "NFLAG_INSURED_ON_APPROVAL"
    ]
}

for table, columns in integer_columns.items():

    print(f"\n========== {table} ==========")

    df = spark.table(f"credit_risk_silver.`{table}`")

    for column in columns:

        if column in df.columns:

            old_type = dict(df.dtypes)[column]

            # Convert to integer
            df = df.withColumn(
                column,
                F.col(column).cast("int")
            )

            print(
                f"{column}: {old_type} -> int"
            )

    # Save updated table
    df.write \
      .format("delta") \
      .mode("overwrite") \
      .option("overwriteSchema", "true") \
      .saveAsTable(f"credit_risk_silver.`{table}`")


print("\n" + "=" * 70)
print("TRANSFORMATION 4 COMPLETED")
print("=" * 70)


========== application(s) ==========
CNT_FAM_MEMBERS: double -> int

========== bureau ==========

========== bureau_balance ==========

========== credit_card_balance ==========
CNT_DRAWINGS_ATM_CURRENT: double -> int
CNT_DRAWINGS_OTHER_CURRENT: double -> int
CNT_DRAWINGS_POS_CURRENT: double -> int
CNT_INSTALMENT_MATURE_CUM: double -> int

========== installments_payments ==========

========== POS_CASH_balance ==========
CNT_INSTALMENT: double -> int
CNT_INSTALMENT_FUTURE: double -> int

========== previous_application ==========
CNT_PAYMENT: double -> int
NFLAG_INSURED_ON_APPROVAL: double -> int

TRANSFORMATION 4 COMPLETED


In [0]:
# ============================================================
# VALIDATION - TRANSFORMATION 4
# CHECK CORRECTED DATA TYPES
# ============================================================

tables = [
    "application(s)",
    "bureau",
    "bureau_balance",
    "credit_card_balance",
    "installments_payments",
    "POS_CASH_balance",
    "previous_application"
]

columns_to_check = {
    "application(s)": [
        "CNT_FAM_MEMBERS"
    ],

    "bureau": [],

    "bureau_balance": [],

    "credit_card_balance": [
        "CNT_DRAWINGS_ATM_CURRENT",
        "CNT_DRAWINGS_OTHER_CURRENT",
        "CNT_DRAWINGS_POS_CURRENT",
        "CNT_INSTALMENT_MATURE_CUM"
    ],

    "installments_payments": [],

    "POS_CASH_balance": [
        "CNT_INSTALMENT",
        "CNT_INSTALMENT_FUTURE"
    ],

    "previous_application": [
        "CNT_PAYMENT",
        "NFLAG_INSURED_ON_APPROVAL"
    ]
}

all_pass = True

for table in tables:

    print(f"\n========== {table} ==========")

    df = spark.table(f"credit_risk_silver.`{table}`")

    schema = dict(df.dtypes)

    for column in columns_to_check[table]:

        actual_type = schema[column]

        if actual_type == "int":
            print(f"PASS - {column}: {actual_type}")
        else:
            print(
                f"FAIL - {column}: "
                f"expected int, found {actual_type}"
            )
            all_pass = False


print("\n" + "=" * 70)

if all_pass:
    print("T4 VALIDATION PASSED")
else:
    print("T4 VALIDATION FAILED")

print("=" * 70)


========== application(s) ==========
PASS - CNT_FAM_MEMBERS: int

========== bureau ==========

========== bureau_balance ==========

========== credit_card_balance ==========
PASS - CNT_DRAWINGS_ATM_CURRENT: int
PASS - CNT_DRAWINGS_OTHER_CURRENT: int
PASS - CNT_DRAWINGS_POS_CURRENT: int
PASS - CNT_INSTALMENT_MATURE_CUM: int

========== installments_payments ==========

========== POS_CASH_balance ==========
PASS - CNT_INSTALMENT: int
PASS - CNT_INSTALMENT_FUTURE: int

========== previous_application ==========
PASS - CNT_PAYMENT: int
PASS - NFLAG_INSURED_ON_APPROVAL: int

T4 VALIDATION PASSED


In [0]:
# ============================================================
# T5 - CHECK CATEGORICAL VALUES
# ============================================================

from pyspark.sql import functions as F

tables = [
    "application(s)",
    "bureau",
    "bureau_balance",
    "credit_card_balance",
    "POS_CASH_balance",
    "previous_application"
]

for table in tables:

    print(f"\n{'=' * 70}")
    print(f"TABLE: {table}")
    print(f"{'=' * 70}")

    df = spark.table(f"credit_risk_silver.`{table}`")

    # Get all string/categorical columns
    string_columns = [
        column for column, dtype in df.dtypes
        if dtype == "string"
    ]

    for column in string_columns:

        print(f"\n--- {column} ---")

        values = (
            df.select(column)
              .filter(F.col(column).isNotNull())
              .groupBy(column)
              .count()
              .orderBy(F.desc("count"))
        )

        display(values)


TABLE: application(s)

--- NAME_CONTRACT_TYPE ---


NAME_CONTRACT_TYPE,count
Cash loans,278232
Revolving loans,29279



--- CODE_GENDER ---


CODE_GENDER,count
F,202448
M,105059
XNA,4



--- FLAG_OWN_CAR ---


...

NAME_SELLER_INDUSTRY,count
XNA,855720
Consumer electronics,398265
Connectivity,276029
Furniture,57849
Construction,29781
Clothing,23949
Industry,19194
Auto technology,4990
Jewelry,2709
MLM partners,1215



--- NAME_YIELD_GROUP ---


NAME_YIELD_GROUP,count
XNA,517215
middle,385532
high,353331
low_normal,322095
low_action,92041



--- PRODUCT_COMBINATION ---


PRODUCT_COMBINATION,count
Cash,286336
POS household with interest,263622
POS mobile with interest,220670
Cash X-Sell: middle,143883
Cash X-Sell: low,130248
Card Street,112582
POS industry with interest,98833
POS household without interest,82908
Card X-Sell,80582
Cash Street: high,59639


In [0]:
# ============================================================
# CHECK ACTUAL CATEGORICAL INCONSISTENCIES
# ============================================================

from pyspark.sql import functions as F

tables = [
    "application(s)",
    "bureau",
    "bureau_balance",
    "credit_card_balance",
    "POS_CASH_balance",
    "previous_application"
]

for table in tables:

    print(f"\n{'=' * 70}")
    print(f"TABLE: {table}")
    print(f"{'=' * 70}")

    df = spark.table(f"credit_risk_silver.`{table}`")

    string_columns = [
        column for column, dtype in df.dtypes
        if dtype == "string"
    ]

    for column in string_columns:

        # Check whitespace differences
        whitespace_issues = (
            df
            .filter(
                F.col(column).isNotNull() &
                (F.col(column) != F.trim(F.col(column)))
            )
            .select(column)
            .distinct()
        )

        whitespace_count = whitespace_issues.count()

        # Check casing variations
        casing_groups = (
            df
            .filter(F.col(column).isNotNull())
            .select(
                F.lower(F.trim(F.col(column))).alias("normalized"),
                F.trim(F.col(column)).alias("actual")
            )
            .distinct()
            .groupBy("normalized")
            .agg(
                F.countDistinct("actual").alias("different_forms")
            )
            .filter(F.col("different_forms") > 1)
        )

        casing_count = casing_groups.count()

        if whitespace_count > 0 or casing_count > 0:

            print(f"\n{column}")

            if whitespace_count > 0:
                print(
                    f"  Whitespace inconsistencies: "
                    f"{whitespace_count}"
                )
                display(whitespace_issues)

            if casing_count > 0:
                print(
                    f"  Casing inconsistencies: "
                    f"{casing_count}"
                )
                display(casing_groups)

        else:
            print(f"{column}: No formatting inconsistencies")

print("\n" + "=" * 70)
print("CATEGORICAL CONSISTENCY CHECK COMPLETED")
print("=" * 70)


TABLE: application(s)
NAME_CONTRACT_TYPE: No formatting inconsistencies
CODE_GENDER: No formatting inconsistencies
FLAG_OWN_CAR: No formatting inconsistencies
FLAG_OWN_REALTY: No formatting inconsistencies
NAME_TYPE_SUITE: No formatting inconsistencies
NAME_INCOME_TYPE: No formatting inconsistencies
NAME_EDUCATION_TYPE: No formatting inconsistencies
NAME_FAMILY_STATUS: No formatting inconsistencies
NAME_HOUSING_TYPE: No formatting inconsistencies
OCCUPATION_TYPE: No formatting inconsistencies
WEEKDAY_APPR_PROCESS_START: No formatting inconsistencies
ORGANIZATION_TYPE: No formatting inconsistencies
EMERGENCYSTATE_MODE: No formatting inconsistencies

TABLE: bureau
CREDIT_ACTIVE: No formatting inconsistencies
CREDIT_CURRENCY: No formatting inconsistencies
CREDIT_TYPE: No formatting inconsistencies

TABLE: bureau_balance
STATUS: No formatting inconsistencies

TABLE: credit_card_balance
NAME_CONTRACT_STATUS: No formatting inconsistencies

TABLE: POS_CASH_balance
NAME_CONTRACT_STATUS: No fo

In [0]:
# ============================================================
# T5 - STANDARDIZE CATEGORICAL VALUES
# ============================================================

from pyspark.sql import functions as F

tables = [
    "application(s)",
    "bureau",
    "bureau_balance",
    "credit_card_balance",
    "POS_CASH_balance",
    "previous_application"
]

for table in tables:

    print(f"\n========== {table} ==========")

    df = spark.table(f"credit_risk_silver.`{table}`")

    string_columns = [
        column for column, dtype in df.dtypes
        if dtype == "string"
    ]

    for column in string_columns:

        df = df.withColumn(
            column,
            F.trim(
                F.regexp_replace(
                    F.col(column),
                    r"\s+",
                    " "
                )
            )
        )

        print(f"Standardized: {column}")

    # Save updated Silver table
    df.write \
      .format("delta") \
      .mode("overwrite") \
      .option("overwriteSchema", "true") \
      .saveAsTable(f"credit_risk_silver.`{table}`")


print("\n" + "=" * 70)
print("TRANSFORMATION 5 COMPLETED")
print("=" * 70)


========== application(s) ==========
Standardized: NAME_CONTRACT_TYPE
Standardized: CODE_GENDER
Standardized: FLAG_OWN_CAR
Standardized: FLAG_OWN_REALTY
Standardized: NAME_TYPE_SUITE
Standardized: NAME_INCOME_TYPE
Standardized: NAME_EDUCATION_TYPE
Standardized: NAME_FAMILY_STATUS
Standardized: NAME_HOUSING_TYPE
Standardized: OCCUPATION_TYPE
Standardized: WEEKDAY_APPR_PROCESS_START
Standardized: ORGANIZATION_TYPE
Standardized: EMERGENCYSTATE_MODE

========== bureau ==========
Standardized: CREDIT_ACTIVE
Standardized: CREDIT_CURRENCY
Standardized: CREDIT_TYPE

========== bureau_balance ==========
Standardized: STATUS

========== credit_card_balance ==========
Standardized: NAME_CONTRACT_STATUS

========== POS_CASH_balance ==========
Standardized: NAME_CONTRACT_STATUS

========== previous_application ==========
Standardized: NAME_CONTRACT_TYPE
Standardized: WEEKDAY_APPR_PROCESS_START
Standardized: FLAG_LAST_APPL_PER_CONTRACT
Standardized: NAME_CASH_LOAN_PURPOSE
Standardized: NAME_CONTRAC

In [0]:
# ============================================================
# T5 VALIDATION - STANDARDIZE CATEGORICAL VALUES
# ============================================================

from pyspark.sql import functions as F

tables = [
    "application(s)",
    "bureau",
    "bureau_balance",
    "credit_card_balance",
    "POS_CASH_balance",
    "previous_application"
]

all_pass = True

for table in tables:

    print(f"\n========== {table} ==========")

    df = spark.table(f"credit_risk_silver.`{table}`")

    string_columns = [
        column for column, dtype in df.dtypes
        if dtype == "string"
    ]

    table_pass = True

    for column in string_columns:

        # Check leading/trailing spaces
        whitespace_count = df.filter(
            F.col(column).isNotNull() &
            (F.col(column) != F.trim(F.col(column)))
        ).count()

        # Check repeated spaces inside values
        repeated_space_count = df.filter(
            F.col(column).isNotNull() &
            F.col(column).rlike(r"\s{2,}")
        ).count()

        if whitespace_count > 0 or repeated_space_count > 0:

            print(
                f"FAIL - {column}: "
                f"leading/trailing spaces = {whitespace_count}, "
                f"repeated spaces = {repeated_space_count}"
            )

            table_pass = False
            all_pass = False

    if table_pass:
        print("PASS - All categorical values are standardized.")


print("\n" + "=" * 70)

if all_pass:
    print("T5 VALIDATION PASSED")
else:
    print("T5 VALIDATION FAILED")

print("=" * 70)


========== application(s) ==========
PASS - All categorical values are standardized.

========== bureau ==========
PASS - All categorical values are standardized.

========== bureau_balance ==========
PASS - All categorical values are standardized.

========== credit_card_balance ==========
PASS - All categorical values are standardized.

========== POS_CASH_balance ==========
PASS - All categorical values are standardized.

========== previous_application ==========
PASS - All categorical values are standardized.

T5 VALIDATION PASSED


In [0]:
# ============================================================
# T6 - CHECK HISTORICAL DAY / MONTH VALUES
# ============================================================

from pyspark.sql import functions as F

historical_columns = {
    "application(s)": [
        "DAYS_BIRTH",
        "DAYS_EMPLOYED",
        "DAYS_REGISTRATION",
        "DAYS_ID_PUBLISH",
        "DAYS_LAST_PHONE_CHANGE"
    ],

    "bureau": [
        "DAYS_CREDIT",
        "DAYS_CREDIT_ENDDATE",
        "DAYS_ENDDATE_FACT",
        "DAYS_CREDIT_UPDATE"
    ],

    "bureau_balance": [
        "MONTHS_BALANCE"
    ],

    "credit_card_balance": [
        "MONTHS_BALANCE"
    ],

    "installments_payments": [
        "DAYS_INSTALMENT",
        "DAYS_ENTRY_PAYMENT"
    ],

    "POS_CASH_balance": [
        "MONTHS_BALANCE"
    ],

    "previous_application": [
        "DAYS_DECISION",
        "DAYS_FIRST_DRAWING",
        "DAYS_FIRST_DUE",
        "DAYS_LAST_DUE_1ST_VERSION",
        "DAYS_LAST_DUE",
        "DAYS_TERMINATION"
    ]
}

for table, columns in historical_columns.items():

    print(f"\n{'=' * 70}")
    print(f"TABLE: {table}")
    print(f"{'=' * 70}")

    df = spark.table(f"credit_risk_silver.`{table}`")

    for column in columns:

        if column in df.columns:

            stats = (
                df.select(
                    F.min(column).alias("minimum"),
                    F.max(column).alias("maximum"),
                    F.avg(column).alias("average")
                )
                .collect()[0]
            )

            print(
                f"{column}: "
                f"min={stats['minimum']}, "
                f"max={stats['maximum']}, "
                f"avg={stats['average']:.2f}"
            )

print("\n" + "=" * 70)
print("HISTORICAL DAY/MONTH CHECK COMPLETED")
print("=" * 70)


TABLE: application(s)
DAYS_BIRTH: min=-25229, max=-7489, avg=-16037.00
DAYS_EMPLOYED: min=-17912, max=365243, avg=63815.05
DAYS_REGISTRATION: min=-24672.0, max=0.0, avg=-4986.12
DAYS_ID_PUBLISH: min=-7197, max=0, avg=-2994.20
DAYS_LAST_PHONE_CHANGE: min=-4292.0, max=0.0, avg=-962.86

TABLE: bureau
DAYS_CREDIT: min=-2922, max=0, avg=-1142.11
DAYS_CREDIT_ENDDATE: min=-42060.0, max=31199.0, avg=458.21
DAYS_ENDDATE_FACT: min=-42023.0, max=0.0, avg=-978.14
DAYS_CREDIT_UPDATE: min=-41947, max=372, avg=-593.75

TABLE: bureau_balance
MONTHS_BALANCE: min=-96, max=0, avg=-30.74

TABLE: credit_card_balance
MONTHS_BALANCE: min=-96, max=-1, avg=-34.52

TABLE: installments_payments
DAYS_INSTALMENT: min=-2922.0, max=-1.0, avg=-1042.27
DAYS_ENTRY_PAYMENT: min=-4921.0, max=-1.0, avg=-1051.07

TABLE: POS_CASH_balance
MONTHS_BALANCE: min=-96, max=-1, avg=-35.01

TABLE: previous_application
DAYS_DECISION: min=-2922, max=-1, avg=-880.68
DAYS_FIRST_DRAWING: min=-2922.0, max=365243.0, avg=351491.78
DAYS_FIR

In [0]:
# ============================================================
# T6 - HANDLE HISTORICAL DAY / MONTH VALUES
# ============================================================

from pyspark.sql import functions as F

historical_columns = {
    "application(s)": [
        "DAYS_BIRTH",
        "DAYS_EMPLOYED",
        "DAYS_REGISTRATION",
        "DAYS_ID_PUBLISH",
        "DAYS_LAST_PHONE_CHANGE"
    ],

    "bureau": [
        "DAYS_CREDIT",
        "DAYS_CREDIT_ENDDATE",
        "DAYS_ENDDATE_FACT",
        "DAYS_CREDIT_UPDATE"
    ],

    "bureau_balance": [
        "MONTHS_BALANCE"
    ],

    "credit_card_balance": [
        "MONTHS_BALANCE"
    ],

    "installments_payments": [
        "DAYS_INSTALMENT",
        "DAYS_ENTRY_PAYMENT"
    ],

    "POS_CASH_balance": [
        "MONTHS_BALANCE"
    ],

    "previous_application": [
        "DAYS_DECISION",
        "DAYS_FIRST_DRAWING",
        "DAYS_FIRST_DUE",
        "DAYS_LAST_DUE_1ST_VERSION",
        "DAYS_LAST_DUE",
        "DAYS_TERMINATION"
    ]
}

SENTINEL_VALUE = 365243

for table, columns in historical_columns.items():

    print(f"\n========== {table} ==========")

    df = spark.table(f"credit_risk_silver.`{table}`")

    for column in columns:

        if column in df.columns:

            # Replace sentinel value with NULL
            df = df.withColumn(
                column,
                F.when(
                    F.col(column) == SENTINEL_VALUE,
                    None
                ).otherwise(F.col(column))
            )

            # Convert historical offsets to positive elapsed values
            df = df.withColumn(
                column,
                F.abs(F.col(column))
            )

            print(f"Handled: {column}")

    # Save updated Silver table
    df.write \
      .format("delta") \
      .mode("overwrite") \
      .option("overwriteSchema", "true") \
      .saveAsTable(f"credit_risk_silver.`{table}`")


print("\n" + "=" * 70)
print("TRANSFORMATION 6 COMPLETED")
print("=" * 70)



========== application(s) ==========
Handled: DAYS_BIRTH
Handled: DAYS_EMPLOYED
Handled: DAYS_REGISTRATION
Handled: DAYS_ID_PUBLISH
Handled: DAYS_LAST_PHONE_CHANGE

========== bureau ==========
Handled: DAYS_CREDIT
Handled: DAYS_CREDIT_ENDDATE
Handled: DAYS_ENDDATE_FACT
Handled: DAYS_CREDIT_UPDATE

========== bureau_balance ==========
Handled: MONTHS_BALANCE

========== credit_card_balance ==========
Handled: MONTHS_BALANCE

========== installments_payments ==========
Handled: DAYS_INSTALMENT
Handled: DAYS_ENTRY_PAYMENT

========== POS_CASH_balance ==========
Handled: MONTHS_BALANCE

========== previous_application ==========
Handled: DAYS_DECISION
Handled: DAYS_FIRST_DRAWING
Handled: DAYS_FIRST_DUE
Handled: DAYS_LAST_DUE_1ST_VERSION
Handled: DAYS_LAST_DUE
Handled: DAYS_TERMINATION

TRANSFORMATION 6 COMPLETED


In [0]:
# ============================================================
# T6 VALIDATION - HISTORICAL DAY / MONTH VALUES
# ============================================================

from pyspark.sql import functions as F

historical_columns = {
    "application(s)": [
        "DAYS_BIRTH",
        "DAYS_EMPLOYED",
        "DAYS_REGISTRATION",
        "DAYS_ID_PUBLISH",
        "DAYS_LAST_PHONE_CHANGE"
    ],

    "bureau": [
        "DAYS_CREDIT",
        "DAYS_CREDIT_ENDDATE",
        "DAYS_ENDDATE_FACT",
        "DAYS_CREDIT_UPDATE"
    ],

    "bureau_balance": [
        "MONTHS_BALANCE"
    ],

    "credit_card_balance": [
        "MONTHS_BALANCE"
    ],

    "installments_payments": [
        "DAYS_INSTALMENT",
        "DAYS_ENTRY_PAYMENT"
    ],

    "POS_CASH_balance": [
        "MONTHS_BALANCE"
    ],

    "previous_application": [
        "DAYS_DECISION",
        "DAYS_FIRST_DRAWING",
        "DAYS_FIRST_DUE",
        "DAYS_LAST_DUE_1ST_VERSION",
        "DAYS_LAST_DUE",
        "DAYS_TERMINATION"
    ]
}

SENTINEL_VALUE = 365243

all_pass = True

for table, columns in historical_columns.items():

    print(f"\n========== {table} ==========")

    df = spark.table(f"credit_risk_silver.`{table}`")

    table_pass = True

    for column in columns:

        if column in df.columns:

            # Check if sentinel value still exists
            sentinel_count = df.filter(
                F.col(column) == SENTINEL_VALUE
            ).count()

            # Check if negative values still exist
            negative_count = df.filter(
                F.col(column) < 0
            ).count()

            if sentinel_count > 0 or negative_count > 0:

                print(
                    f"FAIL - {column}: "
                    f"365243 values = {sentinel_count}, "
                    f"negative values = {negative_count}"
                )

                table_pass = False
                all_pass = False

            else:

                print(
                    f"PASS - {column}: "
                    f"no sentinel/negative values"
                )

    if table_pass:
        print(f"PASS - {table}: Historical values handled correctly.")


print("\n" + "=" * 70)

if all_pass:
    print("T6 VALIDATION PASSED")
else:
    print("T6 VALIDATION FAILED")

print("=" * 70)


========== application(s) ==========
PASS - DAYS_BIRTH: no sentinel/negative values
PASS - DAYS_EMPLOYED: no sentinel/negative values
PASS - DAYS_REGISTRATION: no sentinel/negative values
PASS - DAYS_ID_PUBLISH: no sentinel/negative values
PASS - DAYS_LAST_PHONE_CHANGE: no sentinel/negative values
PASS - application(s): Historical values handled correctly.

========== bureau ==========
PASS - DAYS_CREDIT: no sentinel/negative values
PASS - DAYS_CREDIT_ENDDATE: no sentinel/negative values
PASS - DAYS_ENDDATE_FACT: no sentinel/negative values
PASS - DAYS_CREDIT_UPDATE: no sentinel/negative values
PASS - bureau: Historical values handled correctly.

========== bureau_balance ==========
PASS - MONTHS_BALANCE: no sentinel/negative values
PASS - bureau_balance: Historical values handled correctly.

========== credit_card_balance ==========
PASS - MONTHS_BALANCE: no sentinel/negative values
PASS - credit_card_balance: Historical values handled correctly.

========== installments_payments ===

In [0]:
# ============================================================
# T7 - VALIDATE FINANCIAL VALUES
# COMPLETE CHECK FOR ALL FINANCIAL COLUMNS
# ============================================================

from pyspark.sql import functions as F

financial_columns = {

    "application(s)": [
        "AMT_INCOME_TOTAL",
        "AMT_CREDIT",
        "AMT_ANNUITY",
        "AMT_GOODS_PRICE"
    ],

    "bureau": [
        "AMT_CREDIT_SUM",
        "AMT_CREDIT_SUM_DEBT",
        "AMT_CREDIT_SUM_LIMIT",
        "AMT_CREDIT_SUM_OVERDUE"
    ],

    "credit_card_balance": [
        "AMT_BALANCE",
        "AMT_CREDIT_LIMIT_ACTUAL",
        "AMT_DRAWINGS_ATM_CURRENT",
        "AMT_DRAWINGS_CURRENT",
        "AMT_DRAWINGS_OTHER_CURRENT",
        "AMT_DRAWINGS_POS_CURRENT",
        "AMT_INST_MIN_REGULARITY",
        "AMT_PAYMENT_CURRENT",
        "AMT_PAYMENT_TOTAL_CURRENT",
        "AMT_RECEIVABLE_PRINCIPAL",
        "AMT_RECIVABLE",
        "AMT_TOTAL_RECEIVABLE"
    ],

    "installments_payments": [
        "AMT_INSTALMENT",
        "AMT_PAYMENT"
    ],

    "previous_application": [
        "AMT_ANNUITY",
        "AMT_APPLICATION",
        "AMT_CREDIT",
        "AMT_GOODS_PRICE"
    ]
}

for table, columns in financial_columns.items():

    print(f"\n{'=' * 80}")
    print(f"TABLE: {table}")
    print(f"{'=' * 80}")

    df = spark.table(f"credit_risk_silver.`{table}`")

    for column in columns:

        if column not in df.columns:
            continue

        # Basic statistics
        stats = (
            df.select(
                F.min(column).alias("min_value"),
                F.max(column).alias("max_value"),
                F.avg(column).alias("avg_value"),
                F.expr(f"percentile_approx(`{column}`, 0.5)").alias("median_value")
            )
            .collect()[0]
        )

        # Negative values
        negative_count = df.filter(
            F.col(column) < 0
        ).count()

        # Zero values
        zero_count = df.filter(
            F.col(column) == 0
        ).count()

        # Extremely large values
        # Compare against 99.9th percentile
        percentile_999 = (
            df.select(
                F.expr(
                    f"percentile_approx(`{column}`, 0.999)"
                ).alias("p999")
            )
            .collect()[0]["p999"]
        )

        extreme_count = 0

        if percentile_999 is not None:
            extreme_count = df.filter(
                F.col(column) > percentile_999
            ).count()

        print(f"\n{column}")
        print(f"  Minimum       : {stats['min_value']}")
        print(f"  Maximum       : {stats['max_value']}")
        print(f"  Mean          : {stats['avg_value']}")
        print(f"  Median        : {stats['median_value']}")
        print(f"  Negative      : {negative_count}")
        print(f"  Zero          : {zero_count}")
        print(f"  99.9% value   : {percentile_999}")
        print(f"  Above 99.9%   : {extreme_count}")


print("\n" + "=" * 80)
print("COMPLETE FINANCIAL VALUE CHECK COMPLETED")
print("=" * 80)


TABLE: application(s)

AMT_INCOME_TOTAL
  Minimum       : 25650.0
  Maximum       : 117000000.0
  Mean          : 168797.9192969845
  Median        : 146623.5
  Negative      : 0
  Zero          : 0
  99.9% value   : 900000.0
  Above 99.9%   : 278

AMT_CREDIT
  Minimum       : 45000.0
  Maximum       : 4050000.0
  Mean          : 599025.9997057016
  Median        : 513531.0
  Negative      : 0
  Zero          : 0
  99.9% value   : 2517300.0
  Above 99.9%   : 134

AMT_ANNUITY
  Minimum       : 1615.5
  Maximum       : 258025.5
  Mean          : 27108.497499276447
  Median        : 24907.5
  Negative      : 0
  Zero          : 0
  99.9% value   : 109728.0
  Above 99.9%   : 325

AMT_GOODS_PRICE
  Minimum       : 40500.0
  Maximum       : 4050000.0
  Mean          : 538316.2943667056
  Median        : 450000.0
  Negative      : 0
  Zero          : 0
  99.9% value   : 2250000.0
  Above 99.9%   : 109

TABLE: bureau

AMT_CREDIT_SUM
  Minimum       : 0.0
  Maximum       : 1000000000.0
  Mean 

In [0]:
# ============================================================
# T7 - VALIDATE FINANCIAL VALUES
# HANDLE CLEARLY INVALID FINANCIAL VALUES
# ============================================================

from pyspark.sql import functions as F

# ============================================================
# 1. BUREAU
# ============================================================

df = spark.table("credit_risk_silver.`bureau`")

# Negative debt → NULL
df = df.withColumn(
    "AMT_CREDIT_SUM_DEBT",
    F.when(F.col("AMT_CREDIT_SUM_DEBT") < 0, None)
     .otherwise(F.col("AMT_CREDIT_SUM_DEBT"))
)

# Negative credit limit → NULL
df = df.withColumn(
    "AMT_CREDIT_SUM_LIMIT",
    F.when(F.col("AMT_CREDIT_SUM_LIMIT") < 0, None)
     .otherwise(F.col("AMT_CREDIT_SUM_LIMIT"))
)

# Extreme anomalous value (1 Billion) → NULL
df = df.withColumn(
    "AMT_CREDIT_SUM",
    F.when(F.col("AMT_CREDIT_SUM") == 1000000000, None)
     .otherwise(F.col("AMT_CREDIT_SUM"))
)

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("credit_risk_silver.`bureau`")

print("bureau ✓")


# ============================================================
# 2. CREDIT CARD BALANCE
# ============================================================

df = spark.table("credit_risk_silver.`credit_card_balance`")

# Negative ATM drawing → NULL
df = df.withColumn(
    "AMT_DRAWINGS_ATM_CURRENT",
    F.when(F.col("AMT_DRAWINGS_ATM_CURRENT") < 0, None)
     .otherwise(F.col("AMT_DRAWINGS_ATM_CURRENT"))
)

# Negative drawing amount → NULL
df = df.withColumn(
    "AMT_DRAWINGS_CURRENT",
    F.when(F.col("AMT_DRAWINGS_CURRENT") < 0, None)
     .otherwise(F.col("AMT_DRAWINGS_CURRENT"))
)

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("credit_risk_silver.`credit_card_balance`")

print("credit_card_balance ✓")


print("\n========================================")
print("T7 FINANCIAL TRANSFORMATION COMPLETED")
print("========================================")

bureau ✓
credit_card_balance ✓

T7 FINANCIAL TRANSFORMATION COMPLETED


In [0]:
# ============================================================
# T7 - VALIDATION
# CHECK THAT INVALID FINANCIAL VALUES WERE HANDLED
# ============================================================

from pyspark.sql import functions as F

print("=" * 70)
print("T7 FINANCIAL VALUE VALIDATION")
print("=" * 70)


# ============================================================
# 1. BUREAU VALIDATION
# ============================================================

df = spark.table("credit_risk_silver.`bureau`")

debt_invalid = df.filter(
    F.col("AMT_CREDIT_SUM_DEBT") < 0
).count()

limit_invalid = df.filter(
    F.col("AMT_CREDIT_SUM_LIMIT") < 0
).count()

extreme_credit = df.filter(
    F.col("AMT_CREDIT_SUM") == 1000000000
).count()

print("\nBUREAU")
print("-" * 50)
print("Negative AMT_CREDIT_SUM_DEBT :", debt_invalid)
print("Negative AMT_CREDIT_SUM_LIMIT:", limit_invalid)
print("1 Billion AMT_CREDIT_SUM     :", extreme_credit)


if debt_invalid == 0 and limit_invalid == 0 and extreme_credit == 0:
    print("PASS - Bureau invalid financial values handled")
else:
    print("FAIL - Invalid financial values still exist")


# ============================================================
# 2. CREDIT CARD BALANCE VALIDATION
# ============================================================

df = spark.table("credit_risk_silver.`credit_card_balance`")

atm_invalid = df.filter(
    F.col("AMT_DRAWINGS_ATM_CURRENT") < 0
).count()

drawing_invalid = df.filter(
    F.col("AMT_DRAWINGS_CURRENT") < 0
).count()

print("\nCREDIT CARD BALANCE")
print("-" * 50)
print("Negative AMT_DRAWINGS_ATM_CURRENT:", atm_invalid)
print("Negative AMT_DRAWINGS_CURRENT    :", drawing_invalid)


if atm_invalid == 0 and drawing_invalid == 0:
    print("PASS - Credit card invalid financial values handled")
else:
    print("FAIL - Invalid financial values still exist")


# ============================================================
# FINAL RESULT
# ============================================================

if (
    debt_invalid == 0
    and limit_invalid == 0
    and extreme_credit == 0
    and atm_invalid == 0
    and drawing_invalid == 0
):
    print("\n" + "=" * 70)
    print("T7 VALIDATION PASSED")
    print("All clearly invalid financial values were handled successfully.")
    print("=" * 70)
else:
    print("\n" + "=" * 70)
    print("T7 VALIDATION FAILED")
    print("Some invalid financial values still remain.")
    print("=" * 70)

T7 FINANCIAL VALUE VALIDATION

BUREAU
--------------------------------------------------
Negative AMT_CREDIT_SUM_DEBT : 0
Negative AMT_CREDIT_SUM_LIMIT: 0
1 Billion AMT_CREDIT_SUM     : 0
PASS - Bureau invalid financial values handled

CREDIT CARD BALANCE
--------------------------------------------------
Negative AMT_DRAWINGS_ATM_CURRENT: 0
Negative AMT_DRAWINGS_CURRENT    : 0
PASS - Credit card invalid financial values handled

T7 VALIDATION PASSED
All clearly invalid financial values were handled successfully.


In [0]:
# ============================================================
# T8 - CHECK ALL SILVER TABLES FOR DERIVED FEATURES
# ============================================================

tables = [
    "application(s)",
    "bureau",
    "bureau_balance",
    "credit_card_balance",
    "installments_payments",
    "POS_CASH_balance",
    "previous_application"
]

for table in tables:

    print("\n" + "=" * 80)
    print(f"TABLE: {table}")
    print("=" * 80)

    df = spark.table(f"credit_risk_silver.`{table}`")

    print(f"Total columns: {len(df.columns)}")

    # Show potentially useful numeric/source columns
    relevant_columns = [
        c for c in df.columns
        if (
            c.startswith("AMT_")
            or c.startswith("DAYS_")
            or c.startswith("MONTHS_")
            or c.startswith("CNT_")
            or c.startswith("FLAG_")
            or c.startswith("NFLAG_")
            or c.startswith("RATE_")
        )
    ]

    print("\nPotential source columns:")
    
    for column in relevant_columns:
        print(f"  - {column}")


print("\n" + "=" * 80)
print("T8 ALL-TABLE DERIVED FEATURE CHECK COMPLETED")
print("=" * 80)


TABLE: application(s)
Total columns: 81

Potential source columns:
  - FLAG_OWN_CAR
  - FLAG_OWN_REALTY
  - CNT_CHILDREN
  - AMT_INCOME_TOTAL
  - AMT_CREDIT
  - AMT_ANNUITY
  - AMT_GOODS_PRICE
  - DAYS_BIRTH
  - DAYS_EMPLOYED
  - DAYS_REGISTRATION
  - DAYS_ID_PUBLISH
  - FLAG_MOBIL
  - FLAG_EMP_PHONE
  - FLAG_WORK_PHONE
  - FLAG_CONT_MOBILE
  - FLAG_PHONE
  - FLAG_EMAIL
  - CNT_FAM_MEMBERS
  - DAYS_LAST_PHONE_CHANGE
  - FLAG_DOCUMENT_2
  - FLAG_DOCUMENT_3
  - FLAG_DOCUMENT_4
  - FLAG_DOCUMENT_5
  - FLAG_DOCUMENT_6
  - FLAG_DOCUMENT_7
  - FLAG_DOCUMENT_8
  - FLAG_DOCUMENT_9
  - FLAG_DOCUMENT_10
  - FLAG_DOCUMENT_11
  - FLAG_DOCUMENT_12
  - FLAG_DOCUMENT_13
  - FLAG_DOCUMENT_14
  - FLAG_DOCUMENT_15
  - FLAG_DOCUMENT_16
  - FLAG_DOCUMENT_17
  - FLAG_DOCUMENT_18
  - FLAG_DOCUMENT_19
  - FLAG_DOCUMENT_20
  - FLAG_DOCUMENT_21
  - AMT_REQ_CREDIT_BUREAU_HOUR
  - AMT_REQ_CREDIT_BUREAU_DAY
  - AMT_REQ_CREDIT_BUREAU_WEEK
  - AMT_REQ_CREDIT_BUREAU_MON
  - AMT_REQ_CREDIT_BUREAU_QRT
  - AMT_REQ_CRE

In [0]:
# ============================================================
# T8 - CREATE DERIVED COLUMNS
# ============================================================

from pyspark.sql import functions as F


# ============================================================
# 1. APPLICATION(S)
# ============================================================

df = spark.table("credit_risk_silver.`application(s)`")

# Customer age in years
df = df.withColumn(
    "AGE_YEARS",
    F.round(F.col("DAYS_BIRTH") / 365.25, 0)
)

# Employment duration in years
df = df.withColumn(
    "EMPLOYMENT_YEARS",
    F.round(F.col("DAYS_EMPLOYED") / 365.25, 1)
)

# Credit amount relative to income
df = df.withColumn(
    "CREDIT_INCOME_RATIO",
    F.when(
        F.col("AMT_INCOME_TOTAL") > 0,
        F.col("AMT_CREDIT") / F.col("AMT_INCOME_TOTAL")
    )
)

# Annuity relative to income
df = df.withColumn(
    "ANNUITY_INCOME_RATIO",
    F.when(
        F.col("AMT_INCOME_TOTAL") > 0,
        F.col("AMT_ANNUITY") / F.col("AMT_INCOME_TOTAL")
    )
)

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("credit_risk_silver.`application(s)`")

print("application(s) ✓")


# ============================================================
# 2. BUREAU
# ============================================================

df = spark.table("credit_risk_silver.`bureau`")

# Existing debt relative to credit
df = df.withColumn(
    "DEBT_CREDIT_RATIO",
    F.when(
        F.col("AMT_CREDIT_SUM") > 0,
        F.col("AMT_CREDIT_SUM_DEBT") / F.col("AMT_CREDIT_SUM")
    )
)

# Overdue amount relative to credit
df = df.withColumn(
    "OVERDUE_CREDIT_RATIO",
    F.when(
        F.col("AMT_CREDIT_SUM") > 0,
        F.col("AMT_CREDIT_SUM_OVERDUE") / F.col("AMT_CREDIT_SUM")
    )
)

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("credit_risk_silver.`bureau`")

print("bureau ✓")


# ============================================================
# 3. CREDIT CARD BALANCE
# ============================================================

df = spark.table("credit_risk_silver.`credit_card_balance`")

# Credit card utilization
df = df.withColumn(
    "CREDIT_UTILIZATION_RATIO",
    F.when(
        F.col("AMT_CREDIT_LIMIT_ACTUAL") > 0,
        F.col("AMT_BALANCE") / F.col("AMT_CREDIT_LIMIT_ACTUAL")
    )
)

# Payment relative to balance
df = df.withColumn(
    "PAYMENT_TO_BALANCE_RATIO",
    F.when(
        F.col("AMT_BALANCE") > 0,
        F.col("AMT_PAYMENT_TOTAL_CURRENT") / F.col("AMT_BALANCE")
    )
)

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("credit_risk_silver.`credit_card_balance`")

print("credit_card_balance ✓")


# ============================================================
# 4. INSTALLMENTS PAYMENTS
# ============================================================

df = spark.table("credit_risk_silver.`installments_payments`")

# Difference between actual and scheduled payment
df = df.withColumn(
    "PAYMENT_DIFFERENCE",
    F.col("AMT_PAYMENT") - F.col("AMT_INSTALMENT")
)

# Actual payment relative to scheduled installment
df = df.withColumn(
    "PAYMENT_RATIO",
    F.when(
        F.col("AMT_INSTALMENT") > 0,
        F.col("AMT_PAYMENT") / F.col("AMT_INSTALMENT")
    )
)

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("credit_risk_silver.`installments_payments`")

print("installments_payments ✓")


# ============================================================
# 5. POS CASH BALANCE
# ============================================================

df = spark.table("credit_risk_silver.`POS_CASH_balance`")

# Remaining installments
df = df.withColumn(
    "INSTALLMENT_REMAINING",
    F.col("CNT_INSTALMENT") - F.col("CNT_INSTALMENT_FUTURE")
)

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("credit_risk_silver.`POS_CASH_balance`")

print("POS_CASH_balance ✓")


# ============================================================
# 6. PREVIOUS APPLICATION
# ============================================================

df = spark.table("credit_risk_silver.`previous_application`")

# Approved credit relative to requested application amount
df = df.withColumn(
    "CREDIT_APPLICATION_RATIO",
    F.when(
        F.col("AMT_APPLICATION") > 0,
        F.col("AMT_CREDIT") / F.col("AMT_APPLICATION")
    )
)

# Credit relative to goods price
df = df.withColumn(
    "CREDIT_GOODS_RATIO",
    F.when(
        F.col("AMT_GOODS_PRICE") > 0,
        F.col("AMT_CREDIT") / F.col("AMT_GOODS_PRICE")
    )
)

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("credit_risk_silver.`previous_application`")

print("previous_application ✓")


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 70)
print("T8 DERIVED COLUMNS CREATED SUCCESSFULLY")
print("=" * 70)

application(s) ✓
bureau ✓
credit_card_balance ✓
installments_payments ✓
POS_CASH_balance ✓
previous_application ✓

T8 DERIVED COLUMNS CREATED SUCCESSFULLY


In [0]:
# ============================================================
# T8 - VALIDATION
# CHECK DERIVED COLUMNS
# ============================================================

from pyspark.sql import functions as F

print("=" * 70)
print("T8 DERIVED COLUMNS VALIDATION")
print("=" * 70)


# ============================================================
# 1. APPLICATION(S)
# ============================================================

df = spark.table("credit_risk_silver.`application(s)`")

print("\nAPPLICATION(S)")
print("-" * 50)

print("AGE_YEARS:")
df.select("AGE_YEARS").summary("count", "min", "max", "mean").show()

print("EMPLOYMENT_YEARS:")
df.select("EMPLOYMENT_YEARS").summary("count", "min", "max", "mean").show()

print("CREDIT_INCOME_RATIO:")
df.select("CREDIT_INCOME_RATIO").summary("count", "min", "max", "mean").show()

print("ANNUITY_INCOME_RATIO:")
df.select("ANNUITY_INCOME_RATIO").summary("count", "min", "max", "mean").show()


# ============================================================
# 2. BUREAU
# ============================================================

df = spark.table("credit_risk_silver.`bureau`")

print("\nBUREAU")
print("-" * 50)

df.select(
    "DEBT_CREDIT_RATIO",
    "OVERDUE_CREDIT_RATIO"
).summary("count", "min", "max", "mean").show()


# ============================================================
# 3. CREDIT CARD BALANCE
# ============================================================

df = spark.table("credit_risk_silver.`credit_card_balance`")

print("\nCREDIT CARD BALANCE")
print("-" * 50)

df.select(
    "CREDIT_UTILIZATION_RATIO",
    "PAYMENT_TO_BALANCE_RATIO"
).summary("count", "min", "max", "mean").show()


# ============================================================
# 4. INSTALLMENTS PAYMENTS
# ============================================================

df = spark.table("credit_risk_silver.`installments_payments`")

print("\nINSTALLMENTS PAYMENTS")
print("-" * 50)

df.select(
    "PAYMENT_DIFFERENCE",
    "PAYMENT_RATIO"
).summary("count", "min", "max", "mean").show()


# ============================================================
# 5. POS CASH BALANCE
# ============================================================

df = spark.table("credit_risk_silver.`POS_CASH_balance`")

print("\nPOS CASH BALANCE")
print("-" * 50)

df.select(
    "INSTALLMENT_REMAINING"
).summary("count", "min", "max", "mean").show()


# ============================================================
# 6. PREVIOUS APPLICATION
# ============================================================

df = spark.table("credit_risk_silver.`previous_application`")

print("\nPREVIOUS APPLICATION")
print("-" * 50)

df.select(
    "CREDIT_APPLICATION_RATIO",
    "CREDIT_GOODS_RATIO"
).summary("count", "min", "max", "mean").show()


# ============================================================
# FINAL CHECK - COLUMN EXISTENCE
# ============================================================

expected_columns = {
    "application(s)": [
        "AGE_YEARS",
        "EMPLOYMENT_YEARS",
        "CREDIT_INCOME_RATIO",
        "ANNUITY_INCOME_RATIO"
    ],
    "bureau": [
        "DEBT_CREDIT_RATIO",
        "OVERDUE_CREDIT_RATIO"
    ],
    "credit_card_balance": [
        "CREDIT_UTILIZATION_RATIO",
        "PAYMENT_TO_BALANCE_RATIO"
    ],
    "installments_payments": [
        "PAYMENT_DIFFERENCE",
        "PAYMENT_RATIO"
    ],
    "POS_CASH_balance": [
        "INSTALLMENT_REMAINING"
    ],
    "previous_application": [
        "CREDIT_APPLICATION_RATIO",
        "CREDIT_GOODS_RATIO"
    ]
}

all_pass = True

for table, columns in expected_columns.items():

    df = spark.table(f"credit_risk_silver.`{table}`")

    missing = [c for c in columns if c not in df.columns]

    if missing:
        print(f"{table}: FAIL - Missing {missing}")
        all_pass = False
    else:
        print(f"{table}: PASS - All derived columns present")


# ============================================================
# FINAL RESULT
# ============================================================

if all_pass:
    print("\n" + "=" * 70)
    print("T8 VALIDATION PASSED")
    print("All derived columns were created successfully.")
    print("=" * 70)
else:
    print("\n" + "=" * 70)
    print("T8 VALIDATION FAILED")
    print("=" * 70)

T8 DERIVED COLUMNS VALIDATION

APPLICATION(S)
--------------------------------------------------
AGE_YEARS:
+-------+-----------------+
|summary|        AGE_YEARS|
+-------+-----------------+
|  count|           307511|
|    min|             21.0|
|    max|             69.0|
|   mean|43.90782443554865|
+-------+-----------------+

EMPLOYMENT_YEARS:
+-------+----------------+
|summary|EMPLOYMENT_YEARS|
+-------+----------------+
|  count|          252137|
|    min|             0.0|
|    max|            49.0|
|   mean|6.52744262048023|
+-------+----------------+

CREDIT_INCOME_RATIO:
+-------+--------------------+
|summary| CREDIT_INCOME_RATIO|
+-------+--------------------+
|  count|              307511|
|    min|0.004807615384615385|
|    max|   84.73684210526316|
|   mean|  3.9575702380625097|
+-------+--------------------+

ANNUITY_INCOME_RATIO:
+-------+--------------------+
|summary|ANNUITY_INCOME_RATIO|
+-------+--------------------+
|  count|              307511|
|    min|2.23884

In [0]:
# ============================================================
# T9 - SUMMARIZE HISTORICAL DATA
# STEP 1: CHECK HISTORICAL RECORDS PER CUSTOMER
# ============================================================

from pyspark.sql import functions as F

# Historical tables and their customer-level identifiers
tables = {
    "bureau": "SK_ID_CURR",
    "bureau_balance": "SK_ID_BUREAU",
    "credit_card_balance": "SK_ID_CURR",
    "installments_payments": "SK_ID_CURR",
    "POS_CASH_balance": "SK_ID_CURR",
    "previous_application": "SK_ID_CURR"
}

for table, id_col in tables.items():

    print("\n" + "=" * 80)
    print(f"TABLE: {table}")
    print("=" * 80)

    df = spark.table(f"credit_risk_silver.`{table}`")

    # Total number of records
    total_records = df.count()

    # Number of unique IDs
    unique_ids = df.select(id_col).distinct().count()

    # Number of IDs having multiple historical records
    repeated_ids = (
        df.groupBy(id_col)
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    # Average number of records per ID
    avg_records = total_records / unique_ids if unique_ids > 0 else 0

    print(f"Total records           : {total_records}")
    print(f"Unique {id_col}         : {unique_ids}")
    print(f"Average records per ID  : {avg_records:.2f}")
    print(f"IDs with multiple rows  : {repeated_ids}")


print("\n" + "=" * 80)
print("T9 HISTORICAL DATA CHECK COMPLETED")
print("=" * 80)


TABLE: bureau
Total records           : 1716428
Unique SK_ID_CURR         : 305811
Average records per ID  : 5.61
IDs with multiple rows  : 264291

TABLE: bureau_balance
Total records           : 27299925
Unique SK_ID_BUREAU         : 817395
Average records per ID  : 33.40
IDs with multiple rows  : 811206

TABLE: credit_card_balance
Total records           : 3840312
Unique SK_ID_CURR         : 103558
Average records per ID  : 37.08
IDs with multiple rows  : 102866

TABLE: installments_payments
Total records           : 13605401
Unique SK_ID_CURR         : 339587
Average records per ID  : 40.06
IDs with multiple rows  : 338615

TABLE: POS_CASH_balance
Total records           : 10001358
Unique SK_ID_CURR         : 337252
Average records per ID  : 29.66
IDs with multiple rows  : 336880

TABLE: previous_application
Total records           : 1670214
Unique SK_ID_CURR         : 338857
Average records per ID  : 4.93
IDs with multiple rows  : 278399

T9 HISTORICAL DATA CHECK COMPLETED


In [0]:
# ============================================================
# T9 - SUMMARIZE HISTORICAL DATA
# STEP 2: CREATE CUSTOMER-LEVEL HISTORICAL SUMMARIES
# ============================================================

from pyspark.sql import functions as F


# ============================================================
# 1. BUREAU SUMMARY
# ============================================================

bureau = spark.table("credit_risk_silver.bureau")

bureau_summary = (
    bureau
    .groupBy("SK_ID_CURR")
    .agg(
        F.count("SK_ID_BUREAU").alias("BUREAU_ACCOUNT_COUNT"),
        F.sum("AMT_CREDIT_SUM").alias("BUREAU_TOTAL_CREDIT"),
        F.sum("AMT_CREDIT_SUM_DEBT").alias("BUREAU_TOTAL_DEBT"),
        F.sum("AMT_CREDIT_SUM_OVERDUE").alias("BUREAU_TOTAL_OVERDUE"),
        F.avg("AMT_CREDIT_SUM").alias("BUREAU_AVG_CREDIT"),
        F.avg("AMT_CREDIT_SUM_DEBT").alias("BUREAU_AVG_DEBT"),
        F.max("AMT_CREDIT_SUM").alias("BUREAU_MAX_CREDIT"),
        F.max("CREDIT_DAY_OVERDUE").alias("BUREAU_MAX_DAYS_OVERDUE")
    )
)

bureau_summary.write.format("delta").mode("overwrite").saveAsTable(
    "credit_risk_silver.bureau_summary"
)

print("bureau_summary created")


# ============================================================
# 2. BUREAU BALANCE SUMMARY
# ============================================================

bureau_balance = spark.table("credit_risk_silver.bureau_balance")

bureau_balance_summary = (
    bureau_balance
    .join(
        bureau.select("SK_ID_BUREAU", "SK_ID_CURR").distinct(),
        on="SK_ID_BUREAU",
        how="inner"
    )
    .groupBy("SK_ID_CURR")
    .agg(
        F.count("*").alias("BUREAU_BALANCE_RECORD_COUNT"),
        F.countDistinct("SK_ID_BUREAU").alias("BUREAU_BALANCE_ACCOUNT_COUNT"),
        F.countDistinct("MONTHS_BALANCE").alias("BUREAU_BALANCE_MONTH_COUNT"),
        F.countDistinct("STATUS").alias("BUREAU_BALANCE_STATUS_COUNT")
    )
)

bureau_balance_summary.write.format("delta").mode("overwrite").saveAsTable(
    "credit_risk_silver.bureau_balance_summary"
)

print("bureau_balance_summary created")


# ============================================================
# 3. CREDIT CARD BALANCE SUMMARY
# ============================================================

cc = spark.table("credit_risk_silver.credit_card_balance")

cc_summary = (
    cc
    .groupBy("SK_ID_CURR")
    .agg(
        F.count("SK_ID_PREV").alias("CREDIT_CARD_RECORD_COUNT"),
        F.avg("AMT_BALANCE").alias("CC_AVG_BALANCE"),
        F.max("AMT_BALANCE").alias("CC_MAX_BALANCE"),
        F.avg("AMT_CREDIT_LIMIT_ACTUAL").alias("CC_AVG_CREDIT_LIMIT"),
        F.max("AMT_CREDIT_LIMIT_ACTUAL").alias("CC_MAX_CREDIT_LIMIT"),
        F.avg("CREDIT_UTILIZATION_RATIO").alias("CC_AVG_UTILIZATION"),
        F.max("CREDIT_UTILIZATION_RATIO").alias("CC_MAX_UTILIZATION"),
        F.sum("AMT_PAYMENT_TOTAL_CURRENT").alias("CC_TOTAL_PAYMENT")
    )
)

cc_summary.write.format("delta").mode("overwrite").saveAsTable(
    "credit_risk_silver.credit_card_summary"
)

print("credit_card_summary created")


# ============================================================
# 4. INSTALLMENTS PAYMENTS SUMMARY
# ============================================================

installments = spark.table("credit_risk_silver.installments_payments")

installments_summary = (
    installments
    .groupBy("SK_ID_CURR")
    .agg(
        F.count("SK_ID_PREV").alias("INSTALLMENT_RECORD_COUNT"),
        F.sum("AMT_INSTALMENT").alias("TOTAL_INSTALLMENT_AMOUNT"),
        F.sum("AMT_PAYMENT").alias("TOTAL_PAYMENT_AMOUNT"),
        F.avg("AMT_PAYMENT").alias("AVG_PAYMENT_AMOUNT"),
        F.avg("PAYMENT_RATIO").alias("AVG_PAYMENT_RATIO"),
        F.min("PAYMENT_DIFFERENCE").alias("MIN_PAYMENT_DIFFERENCE"),
        F.max("PAYMENT_DIFFERENCE").alias("MAX_PAYMENT_DIFFERENCE")
    )
)

installments_summary.write.format("delta").mode("overwrite").saveAsTable(
    "credit_risk_silver.installments_summary"
)

print("installments_summary created")


# ============================================================
# 5. POS CASH SUMMARY
# ============================================================

pos = spark.table("credit_risk_silver.POS_CASH_balance")

pos_summary = (
    pos
    .groupBy("SK_ID_CURR")
    .agg(
        F.count("SK_ID_PREV").alias("POS_RECORD_COUNT"),
        F.avg("CNT_INSTALMENT").alias("POS_AVG_INSTALLMENTS"),
        F.max("CNT_INSTALMENT").alias("POS_MAX_INSTALLMENTS"),
        F.avg("CNT_INSTALMENT_FUTURE").alias("POS_AVG_FUTURE_INSTALLMENTS"),
        F.max("SK_DPD").alias("POS_MAX_DPD"),
        F.max("SK_DPD_DEF").alias("POS_MAX_DPD_DEF"),
        F.max("INSTALLMENT_REMAINING").alias("POS_MAX_INSTALLMENT_REMAINING")
    )
)

pos_summary.write.format("delta").mode("overwrite").saveAsTable(
    "credit_risk_silver.pos_cash_summary"
)

print("pos_cash_summary created")


# ============================================================
# 6. PREVIOUS APPLICATION SUMMARY
# ============================================================

previous = spark.table("credit_risk_silver.previous_application")

previous_summary = (
    previous
    .groupBy("SK_ID_CURR")
    .agg(
        F.count("SK_ID_PREV").alias("PREVIOUS_APPLICATION_COUNT"),
        F.sum("AMT_APPLICATION").alias("PREVIOUS_TOTAL_APPLICATION"),
        F.sum("AMT_CREDIT").alias("PREVIOUS_TOTAL_CREDIT"),
        F.avg("AMT_CREDIT").alias("PREVIOUS_AVG_CREDIT"),
        F.max("AMT_CREDIT").alias("PREVIOUS_MAX_CREDIT"),
        F.avg("CREDIT_APPLICATION_RATIO").alias(
            "PREVIOUS_AVG_CREDIT_APPLICATION_RATIO"
        ),
        F.avg("CREDIT_GOODS_RATIO").alias(
            "PREVIOUS_AVG_CREDIT_GOODS_RATIO"
        )
    )
)

previous_summary.write.format("delta").mode("overwrite").saveAsTable(
    "credit_risk_silver.previous_application_summary"
)

print("previous_application_summary created")


# ============================================================
# T9 STEP 2 COMPLETED
# ============================================================

print("\n" + "=" * 80)
print("T9 STEP 2 COMPLETED - ALL HISTORICAL SUMMARIES CREATED")
print("=" * 80)

bureau_summary created
bureau_balance_summary created
credit_card_summary created
installments_summary created
pos_cash_summary created
previous_application_summary created

T9 STEP 2 COMPLETED - ALL HISTORICAL SUMMARIES CREATED


In [0]:
# ============================================================
# T9 - SUMMARIZE HISTORICAL DATA
# STEP 3: VALIDATE HISTORICAL SUMMARIES
# ============================================================

from pyspark.sql import functions as F

summary_tables = {
    "bureau_summary": "SK_ID_CURR",
    "bureau_balance_summary": "SK_ID_CURR",
    "credit_card_summary": "SK_ID_CURR",
    "installments_summary": "SK_ID_CURR",
    "pos_cash_summary": "SK_ID_CURR",
    "previous_application_summary": "SK_ID_CURR"
}

all_pass = True

for table, id_col in summary_tables.items():

    print("\n" + "=" * 80)
    print(f"VALIDATING: {table}")
    print("=" * 80)

    df = spark.table(f"credit_risk_silver.{table}")

    # Total rows
    total_rows = df.count()

    # Unique customer IDs
    unique_ids = df.select(id_col).distinct().count()

    # Duplicate customer IDs
    duplicate_ids = (
        df.groupBy(id_col)
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    # NULL customer IDs
    null_ids = df.filter(F.col(id_col).isNull()).count()

    # Check result
    if total_rows == unique_ids and duplicate_ids == 0 and null_ids == 0:
        status = "PASS"
    else:
        status = "FAIL"
        all_pass = False

    print(f"Total rows             : {total_rows}")
    print(f"Unique SK_ID_CURR      : {unique_ids}")
    print(f"Duplicate customer IDs : {duplicate_ids}")
    print(f"NULL customer IDs      : {null_ids}")
    print(f"Status                 : {status}")


# ============================================================
# FINAL T9 VALIDATION
# ============================================================

print("\n" + "=" * 80)

if all_pass:
    print("T9 VALIDATION PASSED")
    print("All historical summary tables are at customer level.")
else:
    print("T9 VALIDATION FAILED")
    print("Check the summary table showing FAIL.")

print("=" * 80)


VALIDATING: bureau_summary
Total rows             : 305811
Unique SK_ID_CURR      : 305811
Duplicate customer IDs : 0
NULL customer IDs      : 0
Status                 : PASS

VALIDATING: bureau_balance_summary
Total rows             : 134542
Unique SK_ID_CURR      : 134542
Duplicate customer IDs : 0
NULL customer IDs      : 0
Status                 : PASS

VALIDATING: credit_card_summary
Total rows             : 103558
Unique SK_ID_CURR      : 103558
Duplicate customer IDs : 0
NULL customer IDs      : 0
Status                 : PASS

VALIDATING: installments_summary
Total rows             : 339587
Unique SK_ID_CURR      : 339587
Duplicate customer IDs : 0
NULL customer IDs      : 0
Status                 : PASS

VALIDATING: pos_cash_summary
Total rows             : 337252
Unique SK_ID_CURR      : 337252
Duplicate customer IDs : 0
NULL customer IDs      : 0
Status                 : PASS

VALIDATING: previous_application_summary
Total rows             : 338857
Unique SK_ID_CURR      : 

In [0]:
# ============================================================
# T9 - CHECK SUMMARIZED DATA
# ============================================================

from pyspark.sql import functions as F

summary_tables = [
    "bureau_summary",
    "bureau_balance_summary",
    "credit_card_summary",
    "installments_summary",
    "pos_cash_summary",
    "previous_application_summary"
]

for table in summary_tables:

    print("\n" + "=" * 80)
    print(f"SUMMARY TABLE: {table}")
    print("=" * 80)

    df = spark.table(f"credit_risk_silver.{table}")

    print(f"Rows              : {df.count()}")
    print(f"Columns           : {len(df.columns)}")
    print(f"Unique customers  : {df.select('SK_ID_CURR').distinct().count()}")

    print("\nColumns created:")
    print(df.columns)

    print("\nSample summarized records:")
    df.show(5, truncate=False)


SUMMARY TABLE: bureau_summary
Rows              : 305811
Columns           : 9
Unique customers  : 305811

Columns created:
['SK_ID_CURR', 'BUREAU_ACCOUNT_COUNT', 'BUREAU_TOTAL_CREDIT', 'BUREAU_TOTAL_DEBT', 'BUREAU_TOTAL_OVERDUE', 'BUREAU_AVG_CREDIT', 'BUREAU_AVG_DEBT', 'BUREAU_MAX_CREDIT', 'BUREAU_MAX_DAYS_OVERDUE']

Sample summarized records:
+----------+--------------------+-------------------+-----------------+--------------------+-----------------+------------------+-----------------+-----------------------+
|SK_ID_CURR|BUREAU_ACCOUNT_COUNT|BUREAU_TOTAL_CREDIT|BUREAU_TOTAL_DEBT|BUREAU_TOTAL_OVERDUE|BUREAU_AVG_CREDIT|BUREAU_AVG_DEBT   |BUREAU_MAX_CREDIT|BUREAU_MAX_DAYS_OVERDUE|
+----------+--------------------+-------------------+-----------------+--------------------+-----------------+------------------+-----------------+-----------------------+
|207108    |2                   |45000.0            |22005.81         |0.0                 |22500.0          |11002.905         |22500.0

In [0]:
# ============================================================
# FIX CREDIT CARD UTILIZATION NULLS
# ============================================================

from pyspark.sql import functions as F

# Load current Silver credit card table
cc = spark.table("credit_risk_silver.credit_card_balance")

# Recalculate utilization-related columns
cc_fixed = (
    cc
    .withColumn(
        "CREDIT_UTILIZATION_RATIO",
        F.when(
            F.col("AMT_CREDIT_LIMIT_ACTUAL") > 0,
            F.col("AMT_BALANCE") / F.col("AMT_CREDIT_LIMIT_ACTUAL")
        ).otherwise(F.lit(0.0))
    )
)

# Save updated Silver table
cc_fixed.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("credit_risk_silver.credit_card_balance")

print("Credit card utilization values fixed.")


# ============================================================
# RECREATE CREDIT CARD SUMMARY
# ============================================================

cc = spark.table("credit_risk_silver.credit_card_balance")

cc_summary = (
    cc
    .groupBy("SK_ID_CURR")
    .agg(
        F.count("SK_ID_PREV").alias("CREDIT_CARD_RECORD_COUNT"),
        F.avg("AMT_BALANCE").alias("CC_AVG_BALANCE"),
        F.max("AMT_BALANCE").alias("CC_MAX_BALANCE"),
        F.avg("AMT_CREDIT_LIMIT_ACTUAL").alias("CC_AVG_CREDIT_LIMIT"),
        F.max("AMT_CREDIT_LIMIT_ACTUAL").alias("CC_MAX_CREDIT_LIMIT"),
        F.avg("CREDIT_UTILIZATION_RATIO").alias("CC_AVG_UTILIZATION"),
        F.max("CREDIT_UTILIZATION_RATIO").alias("CC_MAX_UTILIZATION"),
        F.sum("AMT_PAYMENT_TOTAL_CURRENT").alias("CC_TOTAL_PAYMENT")
    )
)

cc_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("credit_risk_silver.credit_card_summary")

print("credit_card_summary recreated.")

print("\n" + "=" * 70)
print("CREDIT CARD UTILIZATION FIX COMPLETED")
print("=" * 70)

Credit card utilization values fixed.
credit_card_summary recreated.

CREDIT CARD UTILIZATION FIX COMPLETED


In [0]:
cc_summary = spark.table("credit_risk_silver.credit_card_summary")

cc_summary.select(
    F.count("*").alias("TOTAL_ROWS"),
    F.sum(
        F.when(F.col("CC_AVG_UTILIZATION").isNull(), 1).otherwise(0)
    ).alias("AVG_UTILIZATION_NULLS"),
    F.sum(
        F.when(F.col("CC_MAX_UTILIZATION").isNull(), 1).otherwise(0)
    ).alias("MAX_UTILIZATION_NULLS")
).show()

+----------+---------------------+---------------------+
|TOTAL_ROWS|AVG_UTILIZATION_NULLS|MAX_UTILIZATION_NULLS|
+----------+---------------------+---------------------+
|    103558|                    0|                    0|
+----------+---------------------+---------------------+



In [0]:
# ============================================================
# T10 - JOIN RELATED DATA
# STEP 1: CREATE INTEGRATED CUSTOMER-LEVEL TABLE
# ============================================================

from pyspark.sql import functions as F

# ------------------------------------------------------------
# 1. Load base application table
# ------------------------------------------------------------

application = spark.table(
    "credit_risk_silver.`application(s)`"
)


# ------------------------------------------------------------
# 2. Load all historical summary tables
# ------------------------------------------------------------

bureau_summary = spark.table(
    "credit_risk_silver.bureau_summary"
)

bureau_balance_summary = spark.table(
    "credit_risk_silver.bureau_balance_summary"
)

credit_card_summary = spark.table(
    "credit_risk_silver.credit_card_summary"
)

installments_summary = spark.table(
    "credit_risk_silver.installments_summary"
)

pos_summary = spark.table(
    "credit_risk_silver.pos_cash_summary"
)

previous_summary = spark.table(
    "credit_risk_silver.previous_application_summary"
)


# ------------------------------------------------------------
# 3. Join all summaries to application(s)
# ------------------------------------------------------------

customer_credit_risk = (
    application

    .join(
        bureau_summary,
        on="SK_ID_CURR",
        how="left"
    )

    .join(
        bureau_balance_summary,
        on="SK_ID_CURR",
        how="left"
    )

    .join(
        credit_card_summary,
        on="SK_ID_CURR",
        how="left"
    )

    .join(
        installments_summary,
        on="SK_ID_CURR",
        how="left"
    )

    .join(
        pos_summary,
        on="SK_ID_CURR",
        how="left"
    )

    .join(
        previous_summary,
        on="SK_ID_CURR",
        how="left"
    )
)


# ------------------------------------------------------------
# 4. Save integrated Silver table
# ------------------------------------------------------------

customer_credit_risk.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "credit_risk_silver.customer_credit_risk"
    )


# ------------------------------------------------------------
# 5. Display result
# ------------------------------------------------------------

print("=" * 80)
print("T10 STEP 1 COMPLETED")
print("=" * 80)

print(f"Total rows    : {customer_credit_risk.count()}")
print(f"Total columns : {len(customer_credit_risk.columns)}")

print("\nSample joined data:")

customer_credit_risk.select(
    "SK_ID_CURR",
    "TARGET",
    "BUREAU_ACCOUNT_COUNT",
    "BUREAU_TOTAL_DEBT",
    "CREDIT_CARD_RECORD_COUNT",
    "INSTALLMENT_RECORD_COUNT",
    "POS_RECORD_COUNT",
    "PREVIOUS_APPLICATION_COUNT"
).show(10, truncate=False)

T10 STEP 1 COMPLETED
Total rows    : 307511
Total columns : 126

Sample joined data:
+----------+------+--------------------+-----------------+------------------------+------------------------+----------------+--------------------------+
|SK_ID_CURR|TARGET|BUREAU_ACCOUNT_COUNT|BUREAU_TOTAL_DEBT|CREDIT_CARD_RECORD_COUNT|INSTALLMENT_RECORD_COUNT|POS_RECORD_COUNT|PREVIOUS_APPLICATION_COUNT|
+----------+------+--------------------+-----------------+------------------------+------------------------+----------------+--------------------------+
|100003    |0     |4                   |0.0              |NULL                    |25                      |28              |3                         |
|100004    |0     |2                   |0.0              |NULL                    |3                       |4               |1                         |
|100008    |0     |3                   |240057.0         |NULL                    |35                      |83              |5                        

In [0]:
from pyspark.sql import functions as F

# Load integrated table
df = spark.table(
    "credit_risk_silver.customer_credit_risk"
)

# Expected values from application(s)
application = spark.table(
    "credit_risk_silver.`application(s)`"
)

expected_rows = application.count()

# Validation checks
total_rows = df.count()
unique_customers = df.select("SK_ID_CURR").distinct().count()
duplicate_customers = (
    df.groupBy("SK_ID_CURR")
      .count()
      .filter(F.col("count") > 1)
      .count()
)
null_customer_ids = df.filter(
    F.col("SK_ID_CURR").isNull()
).count()

total_columns = len(df.columns)

print("=" * 80)
print("T10 VALIDATION")
print("=" * 80)

print(f"Expected rows from application(s) : {expected_rows}")
print(f"Rows in customer_credit_risk       : {total_rows}")
print(f"Unique customers                   : {unique_customers}")
print(f"Duplicate customer IDs             : {duplicate_customers}")
print(f"NULL customer IDs                  : {null_customer_ids}")
print(f"Total columns                      : {total_columns}")

if (
    total_rows == expected_rows
    and total_rows == unique_customers
    and duplicate_customers == 0
    and null_customer_ids == 0
):
    print("\nT10 VALIDATION PASSED")
    print("Integrated customer-level table is valid.")
else:
    print("\nT10 VALIDATION FAILED")

T10 VALIDATION
Expected rows from application(s) : 307511
Rows in customer_credit_risk       : 307511
Unique customers                   : 307511
Duplicate customer IDs             : 0
NULL customer IDs                  : 0
Total columns                      : 126

T10 VALIDATION PASSED
Integrated customer-level table is valid.


In [0]:
sfOptions = {
    "host": "NVNAKHZ-NR20168.snowflakecomputing.com",
    "sfUser": "PROJECT1",
    "sfPassword": "Ce7g2UuS8swYMyW",
    "sfDatabase": "CREDIT_RISK_DB",
    "sfSchema": "CREDIT_RISK_GOLD",
    "sfWarehouse": "COMPUTE_WH",
    "sfRole": "ACCOUNTADMIN"
}

print("Serverless Snowflake configuration ready")

Serverless Snowflake configuration ready


In [0]:
silver_df.write \
    .format("snowflake") \
    .options(**sfOptions) \
    .option("dbtable", "CUSTOMER_CREDIT_RISK") \
    .mode("overwrite") \
    .save()

print("SUCCESS: Silver data loaded into Snowflake Gold!")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8744266765852841>, line 1
----> 1 silver_df.write \
      2     .format("snowflake") \
      3     .options(**sfOptions) \
      4     .option("dbtable", "CUSTOMER_CREDIT_RISK") \
      5     .mode("overwrite") \
      6     .save()
      8 print("SUCCESS: Silver data loaded into Snowflake Gold!")

NameError: name 'silver_df' is not defined

In [0]:
df = spark.table("credit_risk_silver.`application(s)`")
df.printSchema()

In [0]:
[c for c in df.columns if "DATE" in c.upper() or "YEAR" in c.upper()]

In [0]:
for table in ["bureau", "previous_application", "installments_payments", "POS_CASH_balance", "credit_card_balance"]:
    try:
        temp = spark.table(f"credit_risk_bronze.`{table}`")
        date_cols = [c for c in temp.columns if "DATE" in c.upper()]
        print(table, "->", date_cols)
    except:
        print(table, "-> table not found")